In [35]:
%load_ext autoreload
%autoreload 2

import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from files import data, features, models

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
# Make sure this matches project root every time

ROOT = '..'

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(ROOT)

os.environ["DATA_DIR"] = "datasets"

In [37]:
listings = data.load("listings")

list_aware = data.load("list_aware")
off_market = data.load("off_market")

In [38]:
homes = listings.sort_values(by = 'date_sold', ascending = True).reset_index(drop = True)

homes.head(3)

,ml_num,sold_price,list_price,date_listed,date_sold,days_on_market,house_type_name,house_style,bedrooms,washrooms,...,house_area,lot_front,lot_depth,municipality,community,latitude,longitude,external_avm_estimate,external_avm_estimate_date,remarks
0,SYN-0023247,364100.0,371200.0,2020-12-29,2021-01-01,3,Condo Apt,Apartment,1,1,...,622,NaN,NaN,Toronto,Sumacpark,43.602039,-79.261184,366200.0,2021-02-14,Looking For More Space In A Turnkey Condo? Thi...
1,SYN-0124536,529900.0,514800.0,2020-11-27,2021-01-01,35,Condo Apt,Apartment,2,1,...,600-799,NaN,NaN,Toronto,Hickoryhaven,43.656349,-79.536538,NaN,2020-12-31,Looking for a downtown commuter condo with pot...
2,SYN-0024710,1805700.0,1745700.0,2020-12-19,2021-01-01,13,Detached,Bungalow,4,2,...,2100-2299,25.14,137.68,Toronto,Hickorymills,43.668013,-79.493774,1790700.0,2021-11-28,A bright eat-in kitchen sets the stage for ent...


In [39]:
homes['sold_to_list_price_ratio'] = homes['sold_price'] / homes['list_price']

In [40]:
amenity_cols = ['bedrooms', 'washrooms', 'parking_total', 'garage']

homes['num_amenities'] = homes[amenity_cols].sum(axis = 1)
# homes.drop(columns = amenity_cols, inplace = True)

In [41]:
homes['house_area_sqft'] = features.square_feet(homes['house_area'])
# homes.drop(columns = ['house_area'], inplace = True)

homes['list_price_per_sqft'] = homes['list_price'] / homes['house_area_sqft']
homes['sold_price_per_sqft'] = homes['sold_price'] / homes['house_area_sqft']

homes['lot_size'] = homes['lot_front'] * homes['lot_depth']
# homes.drop(columns = ['lot_front', 'lot_depth'], inplace = True)

homes['lot_to_house_area'] = homes['lot_size'] / homes['house_area_sqft']
homes['bedrooms_per_sqft'] = homes['bedrooms'] / homes['house_area_sqft']
homes['washrooms_per_sqft'] = homes['washrooms'] / homes['house_area_sqft']
homes['washrooms_to_bedrooms'] = homes['washrooms'] / homes['bedrooms'].replace(0, np.nan)
homes['garage_to_parking'] = homes['garage'] / homes['parking_total'].replace(0, np.nan)

In [42]:
homes['avm_estimate'] = homes[homes['external_avm_estimate_date'] < homes['date_sold']]['external_avm_estimate']
homes.drop(columns = ['external_avm_estimate', 'external_avm_estimate_date'], inplace = True)

In [43]:
homes['list_month_yr'] = homes['date_listed'].dt.to_period('M')
homes['sold_month_yr'] = homes['date_sold'].dt.to_period('M')

In [44]:
earliest_list_date = homes['date_listed'].min()
mean_days_per_month = ((365 * 4) + (366 * 2)) / (12 * 6)

homes['days_since_list_start'] = (homes['date_listed'] - earliest_list_date).dt.days
homes['months_since_list_start'] = homes['days_since_list_start'] / mean_days_per_month

In [45]:
homes['list_month'] = homes['date_listed'].dt.month
homes['list_quart'] = homes['date_listed'].dt.quarter

homes['list_month_sin'] = np.sin(2 * np.pi * homes['list_month'] / 12)
homes['list_month_cos'] = np.cos(2 * np.pi * homes['list_month'] / 12)

homes['list_month_2sin'] = np.sin(4 * np.pi * homes['list_month'] / 12)
homes['list_month_2cos'] = np.cos(4 * np.pi * homes['list_month'] / 12)

homes['list_quart_sin'] = np.sin(2 * np.pi * homes['list_quart'] / 4)
homes['list_quart_cos'] = np.cos(2 * np.pi * homes['list_quart'] / 4)

In [46]:
tor_lon_scale = 0.72

homes['longitude'] *= tor_lon_scale

# homes['lat_cent'] = (homes['latitude'] - homes['latitude'].mean())
# homes['lon_cent'] = (homes['longitude'] - homes['longitude'].mean())

# for deg in [15, 30, 45, 60, 75, 105, 120, 135, 150, 165]:
#     rad = np.deg2rad(deg)
#     homes[f'lat_lon_{deg}'] = (np.cos(rad) * homes['lat_cent'] + np.sin(rad) * homes['lon_cent'])

for deg in [15, 30, 45, 60, 75, 105, 120, 135, 150, 165]:
    rad = np.deg2rad(deg)
    homes[f'lat_lon_{deg}'] = (np.cos(rad) * homes['latitude'] + np.sin(rad) * homes['longitude'])

# homes.drop(columns = ['latitude', 'longitude'], inplace = True)

In [47]:
homes['remarks_count'] = homes['remarks'].str.count(r"\b\w+\b")

remark_kwds = {"renovated": r"renovat|updated|modern|upgrad|remodel",
               "needs_work": r"needs work|fixer|as.is|tlc|original condition",
               "transit": r"subway|transit|t.tc|go station",
               "basement": r"basement|lower level",
               "ravine": r"ravine",
               "view": r"unobstructed|panoramic|city view",
               "balcony": r"balcony|terrace",
               "locker": r"locker|storage unit",
               "move_in": r"move.in ready|turnkey",
               }

for kwd, pattern in remark_kwds.items():
    homes[f'has_{kwd}'] = homes['remarks'].str.lower().str.contains(pattern, regex = True).astype(int)


In [48]:
homes.head(3)

,ml_num,sold_price,list_price,date_listed,date_sold,days_on_market,house_type_name,house_style,bedrooms,washrooms,...,remarks_count,has_renovated,has_needs_work,has_transit,has_basement,has_ravine,has_view,has_balcony,has_locker,has_move_in
0,SYN-0023247,364100.0,371200.0,2020-12-29,2021-01-01,3,Condo Apt,Apartment,1,1,...,51,0,0,0,0,1,1,0,0,1
1,SYN-0124536,529900.0,514800.0,2020-11-27,2021-01-01,35,Condo Apt,Apartment,2,1,...,86,0,1,0,0,0,0,1,0,0
2,SYN-0024710,1805700.0,1745700.0,2020-12-19,2021-01-01,13,Detached,Bungalow,4,2,...,64,0,0,0,0,0,0,0,0,1


In [49]:
# homes['date_listed'].apply(lambda row:
#                            homes[(row - pd.DateOffset(months = 1) <= homes['date_sold']) & (homes['date_sold'] < row)]['sold_price'].median())

In [50]:
list_cols = ['list_price', 'list_price_per_sqft']

# cat_cols = ['house_type_name', 'municipality']
cat_cols = ['house_type_name', 'municipality', 'house_style', 'community']

# loc_cols = []
loc_cols = [col for col in homes if col.startswith('lat_lon_')]
# loc_cols = [col for col in homes if col.startswith('lat_lon_') and col.endswith('0')]
# loc_cols = [col for col in homes if col.startswith('lat_lon_') and col.endswith('5')]

# remark_cols = []
remark_cols = ['remarks_count'] + [col for col in homes if col.startswith('has_')]

# time_angle_cols = []
time_angle_cols = ['list_month_sin', 'list_month_cos']
# time_angle_cols = ['list_month_sin', 'list_month_cos', 'list_quart_sin', 'list_quart_cos']
# time_angle_cols = ['list_month_sin', 'list_month_cos', 'list_month_2sin', 'list_month_2cos', 'list_quart_sin', 'list_quart_cos']

# time_cols = []
# time_cols = ['list_month']
# time_cols = ['list_quart']
time_cols = ['list_month', 'list_quart']

list_aware_cols = (list_cols + cat_cols + loc_cols + remark_cols + amenity_cols + time_angle_cols +
                   ['lot_front', 'lot_depth'] + ['house_area_sqft', 'lot_size', 'months_since_list_start'])

off_market_cols = (cat_cols + loc_cols + remark_cols + amenity_cols + time_cols +
                   ['lot_front', 'lot_depth'] + ['house_area_sqft', 'lot_size', 'months_since_list_start'] +
                   ['lot_to_house_area', 'bedrooms_per_sqft', 'washrooms_per_sqft', 'washrooms_to_bedrooms', 'garage_to_parking'])

In [51]:
results = []

month_periods = pd.period_range("2025-12", "2026-05", freq = "M")

for month in month_periods:
    train, valid = homes[homes['sold_month_yr'] < month], homes[homes['sold_month_yr'] == month]

    X_train, y_train = train[list_aware_cols], train['sold_to_list_price_ratio']
    X_valid, y_valid = valid[list_aware_cols], valid['sold_price']

    gb_model = models.HistGradientBoostingRegressor(random_state = 0,
                                                    categorical_features = cat_cols,
                                                    loss = 'absolute_error', learning_rate = 0.06,
                                                    max_iter = 240, max_leaf_nodes = 30, min_samples_leaf = 40, max_depth = None,
                                                    l2_regularization = 0.1, max_bins = 255, validation_fraction = None,
                                                    max_features = 1., early_stopping = False, verbose = 1)

    gb_model.fit(X_train, np.log(y_train))

    preds = np.exp(gb_model.predict(X_valid)) * X_valid['list_price']
    results.append(pd.DataFrame({'predicted': preds, 'y': y_valid, 'house_type': X_valid['house_type_name']}))
    print('---')

print("List-aware training complete.")

results = pd.concat(results).reset_index(drop = True)

Binning 0.041 GB of training data: 0.406 s
Fitting gradient boosted rounds:
Fit 240 trees in 13.796 s, (7200 total leaves)
Time spent computing histograms: 4.853s
Time spent finding best splits:  1.288s
Time spent applying splits:      1.327s
Time spent predicting:           0.199s
---
Binning 0.042 GB of training data: 0.372 s
Fitting gradient boosted rounds:
Fit 240 trees in 14.916 s, (7200 total leaves)
Time spent computing histograms: 5.402s
Time spent finding best splits:  1.484s
Time spent applying splits:      1.558s
Time spent predicting:           0.208s
---
Binning 0.042 GB of training data: 0.380 s
Fitting gradient boosted rounds:
Fit 240 trees in 13.963 s, (7200 total leaves)
Time spent computing histograms: 4.949s
Time spent finding best splits:  1.316s
Time spent applying splits:      1.379s
Time spent predicting:           0.208s
---
Binning 0.043 GB of training data: 0.393 s
Fitting gradient boosted rounds:
Fit 240 trees in 13.646 s, (7200 total leaves)
Time spent compu

In [52]:
print("MdA%E: " + str(100 * ((results['predicted'] - results['y']).abs() / results['y']).median()))
print("MA%E: " + str(100 * ((results['predicted'] - results['y']).abs() / results['y']).mean()))

print("Signed MdA%E: " + str(100 * ((results['predicted'] - results['y']) / results['y']).median()))
print("Signed MA%E: " + str(100 * ((results['predicted'] - results['y']) / results['y']).mean()))

print('---')

for ht in results['house_type'].unique():
    results_ht = results[results['house_type'] == ht]

    print(f"MdA%E {ht}: " + str(100 * ((results_ht['predicted'] - results_ht['y']).abs() / results_ht['y']).median()))
    print(f"MA%E {ht}: " + str(100 * ((results_ht['predicted'] - results_ht['y']).abs() / results_ht['y']).mean()))

print('---')

print("Baseline MdA%E list-aware: 7.3")
print("Baseline MA%E list_aware: 8.1")

MdA%E: 4.11835572489826
MA%E: 5.248556031956319
Signed MdA%E: 0.3212105580284186
Signed MA%E: -0.13544472800692062
---
MdA%E Detached: 5.091332502583057
MA%E Detached: 6.430128651340253
MdA%E Condo Apt: 3.3865155199424004
MA%E Condo Apt: 4.0319423796574645
MdA%E Freehold Townhouse: 5.596411378657982
MA%E Freehold Townhouse: 6.659247228658437
MdA%E Condo Townhouse: 4.061480956342096
MA%E Condo Townhouse: 5.237548746027103
MdA%E Semi-Detached: 5.636611528779058
MA%E Semi-Detached: 6.847017630237397
MdA%E Co-Op Apt: 4.639961422432454
MA%E Co-Op Apt: 4.928762044305026
---
Baseline MdA%E list-aware: 7.3
Baseline MA%E list_aware: 8.1


In [53]:
results = []

month_periods = pd.period_range("2025-12", "2026-05", freq = "M")

for month in month_periods:
    train, valid = homes[homes['sold_month_yr'] < month], homes[homes['sold_month_yr'] == month]

    X_train, y_train = train[off_market_cols], train['sold_price']
    X_valid, y_valid = valid[off_market_cols], valid['sold_price']

    gb_model = models.HistGradientBoostingRegressor(random_state = 0,
                                                    categorical_features = cat_cols,
                                                    loss = 'squared_error', learning_rate = 0.18,
                                                    max_iter = 400, max_leaf_nodes = 60, min_samples_leaf = 30, max_depth = None,
                                                    l2_regularization = 0.1, max_bins = 255, validation_fraction = None,
                                                    max_features = 0.7, early_stopping = False, verbose = 1)

    gb_model.fit(X_train, np.log(y_train))

    preds = np.exp(gb_model.predict(X_valid))
    results.append(pd.DataFrame({'predicted': preds, 'y': y_valid, 'house_type': X_valid['house_type_name']}))
    print('---')

print("Off-market training complete.")

results = pd.concat(results).reset_index(drop = True)

Binning 0.045 GB of training data: 0.375 s
Fitting gradient boosted rounds:
Fit 400 trees in 18.803 s, (24000 total leaves)
Time spent computing histograms: 7.890s
Time spent finding best splits:  4.155s
Time spent applying splits:      2.717s
Time spent predicting:           0.333s
---
Binning 0.045 GB of training data: 0.374 s
Fitting gradient boosted rounds:
Fit 400 trees in 18.371 s, (24000 total leaves)
Time spent computing histograms: 7.690s
Time spent finding best splits:  4.089s
Time spent applying splits:      2.657s
Time spent predicting:           0.325s
---
Binning 0.046 GB of training data: 0.386 s
Fitting gradient boosted rounds:
Fit 400 trees in 19.553 s, (24000 total leaves)
Time spent computing histograms: 8.168s
Time spent finding best splits:  4.381s
Time spent applying splits:      2.930s
Time spent predicting:           0.333s
---
Binning 0.046 GB of training data: 0.395 s
Fitting gradient boosted rounds:
Fit 400 trees in 218.098 s, (24000 total leaves)
Time spent 

In [54]:
print("MdA%E: " + str(100 * ((results['predicted'] - results['y']).abs() / results['y']).median()))
print("MA%E: " + str(100 * ((results['predicted'] - results['y']).abs() / results['y']).mean()))

print("Signed MdA%E: " + str(100 * ((results['predicted'] - results['y']) / results['y']).median()))
print("Signed MA%E: " + str(100 * ((results['predicted'] - results['y']) / results['y']).mean()))

print('---')

for ht in results['house_type'].unique():
    results_ht = results[results['house_type'] == ht]

    print(f"MdA%E {ht}: " + str(100 * ((results_ht['predicted'] - results_ht['y']).abs() / results_ht['y']).median()))
    print(f"MA%E {ht}: " + str(100 * ((results_ht['predicted'] - results_ht['y']).abs() / results_ht['y']).mean()))

print('---')

print("Baseline MdA%E off-market: 20.8")
print("Baseline MA%E off_market: 27.4")

MdA%E: 7.8563448035434895
MA%E: 10.494165468249172
Signed MdA%E: 0.35698903895789297
Signed MA%E: 1.1455157058170613
---
MdA%E Detached: 10.301138268336404
MA%E Detached: 12.743536143259664
MdA%E Condo Apt: 6.699436991272924
MA%E Condo Apt: 8.973266531329774
MdA%E Freehold Townhouse: 8.819514704434031
MA%E Freehold Townhouse: 11.397434563401587
MdA%E Condo Townhouse: 6.879585504511307
MA%E Condo Townhouse: 8.690232629687896
MdA%E Semi-Detached: 8.948333498847955
MA%E Semi-Detached: 11.217334402225308
MdA%E Co-Op Apt: 10.128515790610775
MA%E Co-Op Apt: 17.54772590009517
---
Baseline MdA%E off-market: 20.8
Baseline MA%E off_market: 27.4
